Vincent Cohen

DSCI 551

3/13/26

# <center> Homework 4: Decoding and Extracting InnoDB Index Records
---

In [1]:
import json
import struct

In [2]:
page_size = 16 * 1024

#The origin of a record points to the beginning of *data* in the record, 
#   not to the beginning of the record itself.
#The infimum record has a fixed size, with header size = 5 bytes.
#   The record header contains metadata about the record, 
#       such as the type of the record.
#   The data of the infimum record is 8 bytes, as shown in class.
#See the following link for more details:
#   https://blog.jcole.us/2013/01/07/the-physical-structure-of-innodb-index-pages/
infimum_record_origin = 94 + 5

In [3]:
def get_index(indexes: list, index_id: int, page_no) -> dict:
    """Returns the index with the given index_id from the list of indexes."""
    for index in indexes:
        if index['index_id'] == index_id and index['root'] == page_no:
            return index

In [4]:
def decode_innodb_float(b: bytes):
    """
    Decode an InnoDB-encoded float (unsigned float is not supported)
    """
    if len(b) != 4:
        raise ValueError("Unsupported float length")

    # innodb stores float in little-endian format
    return struct.unpack('<f', b)[0]

In [5]:
import struct

def decode_innodb_double(b: bytes):
    """
    Decode an InnoDB-encoded double (unsigned double is not supported)
    """
    length = len(b)
    # Standard doubles must be 8 bytes
    if length != 8:
        raise ValueError("double must be exactly 8 bytes")

    # 1. Read the raw bytes as an unsigned 64-bit integer to manipulate the bits
    val = int.from_bytes(b, byteorder='big', signed=False)
    
    # 2. Reverse InnoDB's bit-flipping sorting optimization
    if val & (1 << 63):
        # If the highest bit is 1, it means the original number was positive.
        # We restore it by flipping ONLY the highest bit back to 0.
        val ^= (1 << 63)
    else:
        # If the highest bit is 0, it means the original number was negative.
        # We restore it by flipping ALL bits.
        val ^= 0xFFFFFFFFFFFFFFFF
        
    # 3. Convert the restored bits back to bytes
    restored_bytes = val.to_bytes(8, byteorder='big', signed=False)
    
    # 4. Unpack as a standard IEEE 754 big-endian double
    return struct.unpack('>d', restored_bytes)[0]

In [6]:
def decode_innodb_signed_int(b: bytes) -> int:
    """
    Decode an InnoDB-encoded signed integer of any size:
    1, 2, 3, 4, or 8 bytes.
    """
    length = len(b)
    if length not in (1, 2, 3, 4, 8):
        raise ValueError("Unsupported integer length")

    bits = length * 8

    # Step 1: read as unsigned big-endian
    stored = int.from_bytes(b, byteorder="big", signed=False)

    # Step 2: undo sign-bit flip
    sign_mask = 1 << (bits - 1)
    original = stored ^ sign_mask

    # Step 3: convert to signed range
    if original >= (1 << (bits - 1)):
        original -= (1 << bits)

    return original


In [7]:
def extract_one_record(page, offset, fields):
    """Extracts one record from the page starting at the given offset,
    and returns a dictionary of field values."""

    field_values = {}
    current_offset = offset
    
    for field in fields:
        name = field['name']
        data_type = field['type'].upper()
        size = field['size']
        is_null = field.get('is_null', False)
        
        # Determine if the value is NULL.
        # If the size is 0 and it's allowed to be NULL, it is a NULL value.
        if size == 0:
            if is_null or data_type not in ('VARCHAR', 'TEXT', 'CHAR'):
                field_values[name] = None
                continue
            elif data_type in ('VARCHAR', 'TEXT', 'CHAR'):
                field_values[name] = ""
                continue
                
        # Extract the raw bytes for this field and advance the offset
        raw_bytes = page[current_offset : current_offset + size]
        current_offset += size
        
        # Route to appropriate decoding helpers based on data type
        if 'INT' in data_type: 
            # Handles TINYINT, SMALLINT, MEDIUMINT, INT, BIGINT
            if 'UNSIGNED' in data_type:
                field_values[name] = int.from_bytes(raw_bytes, byteorder='big', signed=False)
            else:
                field_values[name] = decode_innodb_signed_int(raw_bytes)
                
        elif data_type == 'FLOAT':
            field_values[name] = decode_innodb_float(raw_bytes)
            
        elif data_type == 'DOUBLE':
            field_values[name] = decode_innodb_double(raw_bytes)
            
        elif data_type in ('CHAR', 'VARCHAR', 'TEXT'):
            # Decode string; replace bad characters just in case
            field_values[name] = raw_bytes.decode('utf-8', errors='replace')
            
        else:
            # Other data types (like DB_TRX_ID) returned as a hex string
            field_values[name] = '0x' + raw_bytes.hex()
            
    return field_values

In [8]:
def extract_records_for_index(page: bytes, index: dict) -> list:   
    """Extracts all records for the given index from the page, 
        and returns a list of dictionaries of field values."""
    #print('* Index', index['name'])
    
    offset = infimum_record_origin

    # Read 2 bytes for relative offset of first record
    first_rel = int.from_bytes(page[offset-2:offset])  
    #print(f"First relative offset: {first_rel}")
    offset = offset + first_rel  # Move to the first record
    #print(f"First record offset: {offset}")

    record_no = 0
    records = []

    while True:
            rel = int.from_bytes(page[offset-2:offset])  
            
            if rel == 0: # we reach supremum record,
                # which is the last record in the page
                #print("No more records to read.")
                break
            
            # read a record at the current offset
            record = extract_one_record(page, offset, index['records'][record_no])
            #print(record)
            records.append(record)

            record_no += 1
            
            # Move to the next record, with 16-bit wraparound
            offset = (offset + rel) & 0xFFFF  
            #print(f"Next record offset: {offset}")
            
    return records

In [9]:
def extract_indexes_from_ibd(ibd_file: str, decoding_file: str):
    """
    Extract index information from an InnoDB .ibd file using the provided 
    decoding file.The decoding file contains an JSON array of objects where 
    each object shows how to extract records/entries for a specific index. 
    Each object has the following structure:
        {
            "index_id": 565,
            "name": "PRIMARY",
            "records": [
                [ # this is the first record/entry of the index
                    {
                    "name": "id",
                    "type": "int",
                    "size": 4,
                    "is_null": false
                    }, # this is the first field of the first record/entry
                    ...
                ], 
                ...
            ]
        }

    """
    
    with open(decoding_file, 'r') as f:
        indexes = json.load(f)

    index_data = {}

    with open(ibd_file, 'rb') as f:
        while True:
            page = f.read(page_size)  # Read 16KB pages
            if not page:
                break

            #Page type is at offset 24-25
            page_type = int.from_bytes(page[24:26]) 
            #Page no is at offset 4-7
            page_no = int.from_bytes(page[4:8])

            if page_type == 0x45BF:  # This is an index page
                # Extract the index_id from the page header (offset 26-29)
                index_id = int.from_bytes(page[66:74])
                #print('\tindex_id', index_id)
                
                index = get_index(indexes, index_id, page_no)
                if index is None:   # skip this page
                    continue

                records = extract_records_for_index(page, index)
                #print(json.dumps(records, indent=2))

                index_data[index['name']] = records

    return index_data

In [10]:
result = extract_indexes_from_ibd('user.ibd', 'user_decoding.json')
print(json.dumps(result, indent=2))


{
  "PRIMARY": [
    {
      "id": 100,
      "DB_TRX_ID": "0x00000000b019",
      "DB_ROLL_PTR": "0x81000000e20110",
      "name": "0x6a6f686e"
    },
    {
      "id": 101,
      "DB_TRX_ID": "0x00000000b01a",
      "DB_ROLL_PTR": "0x82000000e20110",
      "name": "0x6461766964"
    }
  ],
  "name_idx": [
    {
      "name": "0x6461766964",
      "id": 101
    },
    {
      "name": "0x6a6f686e",
      "id": 100
    }
  ]
}


In [11]:
result = extract_indexes_from_ibd('test.ibd', 'test_decoding.json')
print(json.dumps(result, indent=2))

{
  "PRIMARY": [
    {
      "DB_ROW_ID": "0x0000000dc9fb",
      "DB_TRX_ID": "0x00000000b04e",
      "DB_ROLL_PTR": "0x81000000ff0110",
      "a": 100,
      "b": "0x62696c6c",
      "c": 1
    },
    {
      "DB_ROW_ID": "0x0000000dc9fc",
      "DB_TRX_ID": "0x00000000b04f",
      "DB_ROLL_PTR": "0x82000001210110",
      "a": -463374743,
      "b": "0x6482030000",
      "c": -96
    },
    {
      "DB_ROW_ID": "0x0000000dc9fd",
      "DB_TRX_ID": "0x00000000b054",
      "DB_ROLL_PTR": "0x810000012e0110",
      "a": 67502080,
      "b": "0x",
      "c": -128
    },
    {
      "DB_ROW_ID": "0x0000000dc9fe",
      "DB_TRX_ID": "0x00000000b055",
      "DB_ROLL_PTR": "0x82000001230110",
      "a": 101,
      "b": "0x",
      "c": -27
    }
  ]
}


In [12]:
result = extract_indexes_from_ibd('student.ibd', 'student_decoding.json')
print(json.dumps(result, indent=2))

{
  "PRIMARY": [
    {
      "id": 100,
      "DB_TRX_ID": "0x00000000b037",
      "DB_ROLL_PTR": "0x81000000f40110",
      "name": "0x6a6f686e",
      "gender": "0x6d616c65"
    },
    {
      "id": 101,
      "DB_TRX_ID": "0x00000000b038",
      "DB_ROLL_PTR": "0x82000001530110",
      "name": "0x6d617279",
      "gender": "0x66656d616c65"
    },
    {
      "id": 102,
      "DB_TRX_ID": "0x00000000b03d",
      "DB_ROLL_PTR": "0x81000001570110",
      "name": "0x6461766964",
      "gender": "0x"
    }
  ]
}


In [13]:
result = extract_indexes_from_ibd('test_types.ibd', 'test_types_decoding.json')
print(json.dumps(result, indent=2))

{
  "PRIMARY": [
    {
      "id": 100,
      "DB_TRX_ID": "0x00000000b067",
      "DB_ROLL_PTR": "0x810000011c0110",
      "id1": 101,
      "age": 25,
      "age1": 26,
      "age2": 27,
      "age3": 28,
      "score": 1000,
      "name": "0x6a6f686e20736d697468",
      "gpa": 4.5,
      "salary": -2.35343736894606e-185,
      "height": "0x80af1c",
      "addr": "0x313030206d61706c65207374",
      "dob": "0x8fd422",
      "resume": "my cv is text type"
    }
  ],
  "name": [
    {
      "name": "0x6a6f686e20736d697468",
      "id": 100
    }
  ]
}


In [14]:
result = extract_indexes_from_ibd('tbl1.ibd', 'tbl1_decoding.json')
print(json.dumps(result, indent=2))

{
  "PRIMARY": [
    {
      "a": "0x62696c6c",
      "b": "0x64617669642020202020",
      "DB_TRX_ID": "0x00000000b091",
      "DB_ROLL_PTR": "0x81000001210110",
      "c": -2
    },
    {
      "a": "0x6461766964",
      "b": "0x6a6f686e202020202020",
      "DB_TRX_ID": "0x00000000b08c",
      "DB_ROLL_PTR": "0x820000010f0110",
      "c": -1979449344
    },
    {
      "a": "0x6a6f686e",
      "b": "0x64617669642020202020",
      "DB_TRX_ID": "0x00000000b08b",
      "DB_ROLL_PTR": "0x81000001120110",
      "c": 2
    }
  ]
}


In [15]:
result = extract_indexes_from_ibd('employee.ibd', 'employee_decoding.json')
print(json.dumps(result, indent=2))

{
  "PRIMARY": [
    {
      "id": 100,
      "DB_TRX_ID": "0x00000000b078",
      "DB_ROLL_PTR": "0x810000010b0110",
      "name": "0x6a6f686e202020202020",
      "addr": "0x313030206d61706c6520737472656574"
    },
    {
      "id": 101,
      "DB_TRX_ID": "0x00000000b079",
      "DB_ROLL_PTR": "0x82000001170110",
      "name": "0x62696c6c206368752020",
      "addr": "0x323030207665726d6f6e742061762c204c41"
    },
    {
      "id": 102,
      "DB_TRX_ID": "0x00000000b07a",
      "DB_ROLL_PTR": "0x81000002170110",
      "name": "0x",
      "addr": "0x10313233206d61696e20626c7664"
    }
  ]
}


1. Why are only specific byte lengths supported?Because InnoDB uses standard SQL integer data types that map to specific, fixed physical byte lengths in memory: TINYINT (1 byte), SMALLINT (2 bytes), MEDIUMINT (3 bytes), INT (4 bytes), and BIGINT (8 bytes).
2. Why must the number of bits be computed from the byte length?The exact number of bits is required to locate the position of the sign bit (the most significant bit) and to determine the numerical capacity limits, which is essential for accurate bitwise masking and two's complement arithmetic.
3. Why is the value first read as an unsigned integer?The stored value needs to be read as an unsigned integer first so that bitwise operations can safely be performed to reverse InnoDB's sign-bit flip. If read directly as signed, Python would interpret the flipped bits incorrectly.
4. What does the sign mask (1 << (bits - 1)) represent?It represents a binary mask where only the most significant bit (the sign bit for that specific integer size) is set to 1, and all other bits are 0.
5. What effect does the XOR operation have on the stored value?The XOR operation flips the most significant bit, successfully reverting the storage transformation that InnoDB originally applied to the signed integer.
6. Why might InnoDB flip the sign bit before storing signed integers?InnoDB flips the most significant bit before storage so that negative values sort mathematically correctly before positive values. Without this, a raw byte-by-byte comparison (memcmp) on index pages would erroneously sort negative numbers as larger than positive numbers due to two's complement formatting.
7. Why do we compare against (1 << (bits - 1)) before subtracting?After reverting the sign bit, comparing the value against this mask checks if the restored most significant bit is a 1. If it is, this confirms that the decoded number is meant to be mathematically negative.
8. How does subtracting (1 << bits) convert the value into the correct signed range?In standard two's complement, a negative number is conceptually represented as $2^{\text{bits}} - \text{magnitude}$. By subtracting $2^{\text{bits}}$ (1 << bits), we shift the large unsigned integer interpretation down into its correct arithmetic negative value.
9. What would happen if the sign-bit flip step were skipped?The sorting optimization would remain "baked" into the integer. Positive numbers would be decoded entirely incorrectly as negative numbers, and negative numbers would be decoded as positive numbers.
10. Explain the full transformation from stored bytes to final signed integer.The transformation happens in three steps: First, the raw bytes are read as a standard unsigned integer. Second, the sign bit is flipped using an XOR operation to undo InnoDB's B+Tree sorting optimization. Finally, the result is verified; if the restored sign bit indicates it is a negative number, it is mathematically converted back into the proper signed negative value by subtracting the maximum bit capacity.